## Classical clustering - performance evaluation

In [1]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("ML").master("local[*]").getOrCreate()
spark.conf.set("spark.sql.ansi.enabled", False)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/15 03:33:09 WARN Utils: Your hostname, DESKTOP-UQF5BSK, resolves to a loopback address: 127.0.1.1; using 172.24.225.163 instead (on interface eth0)
26/04/15 03:33:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/15 03:33:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In the first experiments we will ignore the order of logs inside individual sessions, we will one-hot categorical variables, we will also drop country column for now as most of logs are coming from Poland, year contains only one value, month contains only four. Day number will be Min-Max scaled and Price will be whitened, after we will apply PCA

In [2]:
from utils.ml import pipeline, evaluate_clustering_model
df, pipe = pipeline()

+---+-------------+----------------------+------+--------+-----------------+-----+-------+----+------------+------------+
|day|main_category|page 1 (main category)|colour|location|model photography|price|price 2|page|model_letter|model_number|
+---+-------------+----------------------+------+--------+-----------------+-----+-------+----+------------+------------+
|  1|            0|                     1|     1|       5|                1|   28|      2|   1|           A|          13|
|  1|            0|                     1|     1|       6|                1|   33|      2|   1|           A|          16|
|  1|            0|                     2|    10|       2|                1|   52|      1|   1|           B|           4|
|  1|            0|                     2|     6|       6|                2|   38|      2|   1|           B|          17|
|  1|            0|                     2|     4|       3|                2|   52|      1|   1|           B|           8|
+---+-------------+-----

## Evaluation

In [3]:
n_components = [30, 20, 15]
results = {"K-Means": [], "Biscting KMeans": [], "GMM": []}

In [ ]:
from pyspark.ml.clustering import BisectingKMeans, KMeans, GaussianMixture
import matplotlib.pyplot as plt
pca = pipe.getStages()[-1]
for n in n_components:
    pca.setK(n)
    fitted = pipe.fit(df)
    print("Explained Variance:", sum(fitted.stages[-1].explainedVariance))
    X = fitted.transform(df)
    results["K-Means"].append(evaluate_clustering_model(KMeans(maxIter=30), X, range(2, 21)))
    results["Bisecting KMeans"].append(evaluate_clustering_model(BisectingKMeans(maxIter=50), X, range(2, 21)))
    results["GMM"].append(evaluate_clustering_model(GaussianMixture(), X, range(2, 21)))


Explained Variance: 0.8986409801830322


26/04/15 03:35:08 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/15 03:35:13 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
ERROR:root:KeyboardInterrupt while sending command.                 (0 + 2) / 2]
Traceback (most recent call last):
  File "/home/leonid/miniconda3/envs/bigdata_env/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/home/leonid/miniconda3/envs/bigdata_env/lib/python3.10/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/home/leonid/miniconda3/envs/bigdata_env/lib/python3.10/socket.py", line 717, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
fig, ax = plt.subplots(3, 3, figsize=(15, 10), sharex=True)

models = list(results.keys())

for r in range(3):
    for c, v in enumerate(models):
        ax[r, c].plot(results[v][r]["N_K"], results[v][r]["silhouette"], label="Silhouette")
        ax[r, c].plot(results[v][r]["N_K"], results[v][r]["davies_bouldin"], label="DB")
        ax[r, c].plot(results[v][r]["N_K"], results[v][r]["calinski_harabasz"], label="CH")
        if r == 0:
            ax[r, c].set_title(v)
        if c == 0:
            ax[r, c].set_ylabel(f"Setting {r+1}")
        ax[r, c].legend()
        ax[r, c].set_xlabel("Number of Clusters (K)")
plt.tight_layout()
plt.show()